# GRPO Demo

This tutorial demonstrates training the [Gemma](https://deepmind.google/models/gemma/) 
2 2B-IT model on the [GSM8K math reasoning benchmark](https://huggingface.co/datasets/openai/gsm8k) 
using [Group Relative Policy Optimization (GRPO)](https://arxiv.org/pdf/2402.03300). 
GRPO can enhance your model's problem-solving skills on mathematical word problems,
coding problems, etc.

GRPO is an RL algorithm designed to enhance the reasoning abilities of LLMs. It
is a variant of [Proximal Policy Optimization (PPO)](https://arxiv.org/abs/1707.06347) 
that reduces memory usage by eliminating the need for a separate value function
model. GRPO works by generating multiple responses for a given prompt, 
evaluating these responses using a reward model, and then calculating a relative
advantage based on the group's performance to update the policy.

In this tutorial we use a `v5e-8` TPU for Gemma2-2b-it. Let's get started!

Note that the setup below is for the Gemma2-2B-IT model only. If you want to use
another model (say, Qwen2.5), you may need to change the setup (for example, 
tokenizer, chat template, reward function, etc.).

## Install necessary libraries

In [ ]:
!pip install -q wandb
!pip install -q kagglehub

!pip install -q ipywidgets
# Install JupyterLab extension for ipywidgets support (fixes 'ipywidgetsKernel' error)
!pip install -q jupyterlab-widgets
# Enable the extension for classic notebook (if applicable)
!jupyter nbextension enable --py widgetsnbextension --sys-prefix 2>/dev/null || true
# Note: After running this cell, RESTART THE KERNEL for the extension to take effect

!pip install -q tensorflow
!pip install -q tensorflow_datasets
!pip install -q tensorboardX
!pip install -q transformers
!pip install -q grain
!pip install "google-tunix[prod]==0.1.3"

# !pip install -q git+https://github.com/google/tunix
# !pip install -q git+https://github.com/google/qwix

!pip uninstall -q -y flax
# !pip install -U flax
!pip install flax==0.12.2

!pip install -q datasets wandb==0.22.0

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['WANDB_API_KEY'] = os.getenv('WANDB_API_KEY')
os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY')

## Imports

In [ ]:
import functools
import gc
import os
from pprint import pprint
import re

import csv
import shutil

from flax import nnx
import grain
import humanize
import jax
import jax.numpy as jnp
import kagglehub
import optax
from orbax import checkpoint as ocp
from pathlib import Path
import qwix
import tensorflow_datasets as tfds
# from tqdm.auto import tqdm
from tqdm import tqdm
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.rollout import base_rollout
from tunix.sft import metrics_logger

## Hyperparameters

Let's define the configuration we are going to use. Note that this is by no
means a "perfect" set of hyperparameters. To get good results, you might have
to train the model for longer.

In [ ]:
# ====== Data ======
TRAIN_DATA_DIR = "./data/train"
TEST_DATA_DIR = "./data/test"
TRAIN_FRACTION = 1.0

# ====== LoRA ======
RANK = 64
ALPHA = 64.0

# ====== Sharding ======
MESH = [(1, 4), ("fsdp", "tp")]

# ====== GRPO ======
# === Generation during GRPO training ===
MAX_PROMPT_LENGTH = 256
TOTAL_GENERATION_STEPS = 512
# Important to keep a high-ish temperature for varied, diverse responses during
# training.
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = 50
# The number of times the policy generates multiple responses for a given prompt
# within a single training step. This corresponds to `G` in Algorithm 1 in the
# paper. The "group" in GRPO comes from here.
NUM_GENERATIONS = 4

# === other GRPO configs ===
# The number of iterations per batch (𝜇 in GRPO algo 1).
NUM_ITERATIONS = 1
# The coefficient for the KL divergence penalty (𝛽) in the GRPO loss function.
# Important to keep a high enough value for this, otherwise, the KL divergence
# can increase unchecked.
BETA = 0.08
# Epsilon value for clipping (𝜀 in GRPO loss in paper). Similar to PPO, for
# stable updates.
EPSILON = 0.2

# ====== Training ======
TRAIN_MICRO_BATCH_SIZE = 2
# Increase `NUM_BATCHES` and `MAX_STEPS` for better results.
NUM_BATCHES = 3738
# Keep `NUM_TEST_BATCHES` low so that evaluation runs quickly. It can be
# increased to a max. of 330 (if batch size is 4).
NUM_TEST_BATCHES = 100

EVAL_EVERY_N_STEPS = 10  # this doesn't matter if `TRAIN_FRACTION = 1.0`.
NUM_EPOCHS = 1  # can potentially train for more epochs

# Number of training steps.
MAX_STEPS = int(NUM_BATCHES * NUM_ITERATIONS * TRAIN_FRACTION * NUM_EPOCHS)

# === AdamW, warmup, cosine scheduler ===
LEARNING_RATE = 3e-6
B1 = 0.9
B2 = 0.99
WEIGHT_DECAY = 0.1
# == Cosine decay with warmup scheduler ==
# Linearly increase learning rate from 0. to 5e-6 in the first 10% training
# steps, and then gradually decrease the learning rate to 0 using cosine
# scheduler.
WARMUP_STEPS = 0.1 * MAX_STEPS
# == Grad clipping ==
# Grad clipping to prevent large gradients. Found this
# important to keep KL divergence in check.
MAX_GRAD_NORM = 0.1

# Checkpoint saving
INTERMEDIATE_CKPT_DIR = os.path.abspath("./intermediate_ckpt/")
CKPT_DIR = os.path.abspath("./ckpts/")
SAVE_INTERVAL_STEPS = 500
MAX_TO_KEEP = 4

# ====== Inference ======
GENERATION_CONFIGS = {
    # greedy search
    "greedy": {"temperature": 1e-4, "top_k": 1, "top_p": 1.0},
    # some randomness
    "standard": {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
    # liberal
    "liberal": {"temperature": 0.85, "top_k": 2000, "top_p": 1.0},
}

## Utility functions

In [ ]:
def show_hbm_usage():
  """Displays memory usage per device."""
  fmt_size = functools.partial(humanize.naturalsize, binary=True)

  for d in jax.local_devices():
    stats = d.memory_stats()
    used = stats["bytes_in_use"]
    limit = stats["bytes_limit"]
    print(f"Using {fmt_size(used)} / {fmt_size(limit)} ({used/limit:%}) on {d}")

## Data preprocessing

First, let's define some special tokens. We instruct the model to first reason
between the `<reasoning>` and `</reasoning>` tokens. After
reasoning, we expect it to provide the answer between the `<answer>` and
`</answer>` tokens.

In [ ]:
reasoning_start = "<reasoning>"
reasoning_end = "</reasoning>"
solution_start = "<answer>"
solution_end = "</answer>"


SYSTEM_PROMPT = f"""You are given a problem. Think about the problem and \
provide your reasoning. Place it between {reasoning_start} and \
{reasoning_end}. Then, provide the final answer (i.e., just one numerical \
value) between {solution_start} and {solution_end}."""

TEMPLATE = """<start_of_turn>user
{system_prompt}

{question}<end_of_turn>
<start_of_turn>model"""

We use OpenAI's [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k), which comprises grade school math word problems.

In [ ]:
def extract_hash_answer(text: str) -> str | None:
  if "####" not in text:
    return None
  return text.split("####")[1].strip()


def _load_from_tfds(data_dir: str, split: str):
  import tensorflow_datasets.text.gsm8k
  return tfds.data_source(
      "gsm8k",
      split=split,
      data_dir=data_dir,
      builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
      download=True,
  )


def download_kaggle_dataset(target_dir="./data/gsm8k"):
  os.makedirs(target_dir, exist_ok=True)
  src = kagglehub.dataset_download("thedevastator/grade-school-math-8k-q-a")
  src = Path(src)
  dst = Path(target_dir)

  for csv_file in src.glob("*.csv"):  # match all CSV files
    shutil.copy2(csv_file, dst / csv_file.name)
    print(f"Copied {csv_file.name} → {dst/csv_file.name}")
  return target_dir


def get_dataset(data_dir, split="train", source="tfds") -> grain.MapDataset:
  # Download data
  if not os.path.exists(data_dir):
    os.makedirs(data_dir)

  if source == "tfds":
    import tensorflow_datasets.text.gsm8k
    data = tfds.data_source(
        "gsm8k",
        split=split,
        data_dir=data_dir,
        builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
        download=True,
    )

  elif source == "kaggle":
    kaggle_dir = download_kaggle_dataset(data_dir)
    file_name = "main_" + split + ".csv"
    csv_path = os.path.join(kaggle_dir, file_name)  # adjust filename if needed

    data = []
    with open(csv_path, newline="", encoding="utf-8") as csvfile:
      reader = csv.DictReader(csvfile)
      for row in reader:
        data.append({
            "question": row["question"],
            "answer": row["answer"],
        })

  else:
    raise ValueError(f"Unknown source: {source}")

  def _as_text(v):
    return v if isinstance(v, str) else v.decode("utf-8")

  dataset = (
      grain.MapDataset.source(data)
      .shuffle(seed=42)
      .map(
          lambda x: {
              # passed to model forward pass
              "prompts": TEMPLATE.format(
                  system_prompt=SYSTEM_PROMPT,
                  question=_as_text(x["question"]),
              ),
              # passed to reward functions
              "question": _as_text(x["question"]),
              # passed to reward functions
              "answer": extract_hash_answer(_as_text(x["answer"])),
          }
      )
  )
  return dataset

We split the dataset set into train and test sets as usual.

In [ ]:
# source = input("Choose data source [tfds/kaggle]: ").strip().lower()
source = 'kaggle'

if source not in ("tfds", "kaggle"):
  print("Invalid choice. Defaulting to 'tfds'.")
  source = "tfds"

print(f"Using data source: {source}")

dataset = get_dataset(TRAIN_DATA_DIR, "train", source).batch(TRAIN_MICRO_BATCH_SIZE)[
    :NUM_BATCHES
]

if TRAIN_FRACTION == 1.0:
  train_dataset = dataset.repeat(NUM_EPOCHS)
  val_dataset = None
else:
  train_dataset = dataset[: int(len(dataset) * TRAIN_FRACTION)]
  train_dataset = train_dataset.repeat(NUM_EPOCHS)

  val_dataset = dataset[int(len(dataset) * TRAIN_FRACTION) :].repeat(NUM_EPOCHS)

test_dataset = get_dataset(TEST_DATA_DIR, "test", source).batch(TRAIN_MICRO_BATCH_SIZE)[
    :NUM_TEST_BATCHES
]

dataset_lengths = (
    len(train_dataset),
    len(val_dataset) if val_dataset is not None else 0,
    len(test_dataset),
)
print(f"dataset contains {dataset_lengths} of batches")

Let's see how one batch of the training dataset looks like!


In [ ]:
for ele in train_dataset[:1]:
  pprint(ele)

## Load the policy model and the reference model

The policy model is the model which is actually trained and whose weights are
updated. The reference model is the model with which we compute KL divergence.
This is to ensure that the policy updates are not huge and that it does not
deviate too much from the reference model.

Typically, the reference model is the base model, and the policy model is the
same base model, but with LoRA parameters. Only the LoRA parameters are updated.

Note: We perform full precision (fp32) training. You can, however, leverage
Qwix for QAT.

To load the model, you need to be on [Kaggle](https://www.kaggle.com/) and need
to have agreed to the Gemma license
[here](https://www.kaggle.com/models/google/gemma/flax/).

In [ ]:
# Log in
if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
  kagglehub.login()

In [ ]:
model_path = {
    "gemma2": "google/gemma-2/flax/",
}
model_family = "gemma2"
model_version = "gemma2-2b-it"
print(f"{model_path[model_family]}{model_version}")

kaggle_ckpt_path = kagglehub.model_download(
    f"{model_path[model_family]}{model_version}"
)

This code snippet serves as a workaround to re-save the pre-trained model checkpoint from Kaggle into a local format that is compatible with the [Flax NNX](https://flax.readthedocs.io/en/stable/why.html) library. Because the original checkpoint has parameter names and tensor structures that don't match the target NNX model architecture, it cannot be loaded directly.

We first load the original weights into a temporary model instance, then extract and re-save the model's state into a new, properly formatted local checkpoint, which can then be successfully loaded by the final sharded NNX model.

In [ ]:
!rm ./intermediate_ckpt/* -rf

!rm ./ckpts/* -rf

if model_family == "gemma2":
  params = params_lib.load_and_format_params(
      os.path.join(kaggle_ckpt_path, "gemma2-2b-it")
  )
  gemma = gemma_lib.Transformer.from_params(params, version="2-2b-it")
  checkpointer = ocp.StandardCheckpointer()
  _, state = nnx.split(gemma)
  checkpointer.save(os.path.join(INTERMEDIATE_CKPT_DIR, "state"), state)
  checkpointer.wait_until_finished()
  # Delete the intermediate model to save memory.
  del params
  del gemma
  del state
  gc.collect()

### Model Loading and LoRA Application

These two functions work together to load a base model from a checkpoint and apply a LoRA (Low-Rank Adaptation) layer to it.

* `get_ref_model`: Loads the complete Gemma model from a specified checkpoint path. It uses **JAX sharding** to distribute the model parameters across multiple devices.
* `get_lora_model`: Takes the base model and applies LoRA layers to it. It uses a `LoraProvider` to select specific layers (like attention and MLP layers) to be adapted. The resulting LoRA-infused model is then sharded and updated to ensure it's ready for distributed training.

In [ ]:
def get_gemma_ref_model(ckpt_path):
  mesh = jax.make_mesh(*MESH)
  model_config = gemma_lib.ModelConfig.gemma2_2b()
  abs_gemma: nnx.Module = nnx.eval_shape(
      lambda: gemma_lib.Transformer(model_config, rngs=nnx.Rngs(params=0))
  )
  abs_state = nnx.state(abs_gemma)
  abs_state = jax.tree.map(
      lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
      abs_state,
      nnx.get_named_sharding(abs_state, mesh),
  )
  checkpointer = ocp.StandardCheckpointer()
  restored_params = checkpointer.restore(ckpt_path, target=abs_state)

  graph_def, _ = nnx.split(abs_gemma)
  gemma = nnx.merge(graph_def, restored_params)
  return gemma, mesh, model_config


def get_lora_model(base_model, mesh):
  lora_provider = qwix.LoraProvider(
      module_path=(
          ".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj|"
          ".*attn_vec_einsum"
      ),
      rank=RANK,
      alpha=ALPHA,
  )

  model_input = base_model.get_model_input()
  lora_model = qwix.apply_lora_to_model(
      base_model, lora_provider, **model_input
  )

  with mesh:
    state = nnx.state(lora_model)
    pspecs = nnx.get_partition_spec(state)
    sharded_state = jax.lax.with_sharding_constraint(state, pspecs)
    nnx.update(lora_model, sharded_state)

  return lora_model

Now we load reference and policy Gemma models using the Flax NNX library and display their structures.

In [ ]:
# Reference model
if model_family == "gemma2":
  ref_model, mesh, model_config = get_gemma_ref_model(
      ckpt_path=os.path.join(INTERMEDIATE_CKPT_DIR, "state")
  )

In [ ]:
# Policy model
lora_policy = get_lora_model(ref_model, mesh=mesh)
# nnx.display(lora_policy)

In [ ]:
if model_family == "gemma2":
  tokenizer = tokenizer_lib.Tokenizer(
      tokenizer_path=os.path.join(kaggle_ckpt_path, "tokenizer.model")
  )

## Define reward functions

We define four reward functions:

- reward if the format of the output exactly matches the instruction given in
`TEMPLATE`;
- reward if the format of the output approximately matches the instruction given
in `TEMPLATE`;
- reward if the answer is correct/partially correct;
- Sometimes, the text between `<answer>`, `</answer>` might not be one
  number. So, we extract the number, and reward the model if the answer is correct.

The reward functions are inspired from
[here](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb).

First off, let's define a RegEx for checking whether the format matches.

## PRM Training

In [ ]:
PRM_PLACEHOLDER_TOKEN=" ки"
PRM_MAX_LENGTH=256
PRM_REWARD_TOKENS=["+", "-"]
PRM_BATCH_SIZE=1
PRM_GRADIENT_ACCUMULATOR_STEPS=16
PRM_NUM_EPOCHS=1
PRM_WARMUP_RATIO=0.1
PRM_LEARNING_RATE=1e-5
PRM_PAD_TOKEN_ID=0
PRM_CHECKPOINT_ENTRY=1000

In [ ]:
from datasets import load_dataset
import numpy as np
from flax import serialization

In [ ]:

dataset = load_dataset("zhuzilin/Math-Shepherd", split="train")
split_dataset = dataset.train_test_split(
    test_size=0.79,   # 10% for eval
    seed=42
)
train_data = split_dataset["train"]
eval_data  = split_dataset["test"]

print(len(train_data), len(eval_data))


In [ ]:
def strip_bos_eos(ids, tokenizer):
    if len(ids) > 0 and ids[0] == tokenizer.bos_id():
        ids = ids[1:]
    if len(ids) > 0 and ids[-1] == tokenizer.eos_id():
        ids = ids[:-1]
    return ids

def convert_token_to_id(token, tokenizer):
    token_ids = tokenizer.encode(token)
    return strip_bos_eos(token_ids, tokenizer)[0]

def process_data(in_dataset, placeholder_token, in_tokenizer, max_length):
    placeholder_token_id = convert_token_to_id(placeholder_token, in_tokenizer)
    
    inputs = in_dataset['input']
    labels = in_dataset['value']

    processed_data = []

    for i in tqdm(range(len(inputs)), desc="Processing data"):
        input_text = inputs[i]
        label_values = labels[i]

        token_ids = strip_bos_eos(in_tokenizer.encode(input_text), in_tokenizer)
        
        if len(token_ids) > max_length:
            token_ids = token_ids[:max_length]

        input_ids = jnp.array(token_ids)
        attention_masks = jnp.ones_like(input_ids)

        label_tokens = []
        for label in label_values:
            label_tokens.append(convert_token_to_id(label, in_tokenizer))
        
        label_tensors = jnp.array(label_tokens, dtype=input_ids.dtype)

        mask = input_ids == placeholder_token_id
        num_placeholders = mask.sum()

        truncated_labels = label_tensors[:num_placeholders]

        full_labels = jnp.full_like(input_ids, -100, dtype=label_tensors.dtype)
        full_labels = full_labels.at[mask].set(truncated_labels)

        processed_data.append((
            input_ids,
            attention_masks,
            full_labels
        ))
    
    return processed_data


def padded_data(batch, pad_token_id):
    input_ids = []
    attention_masks = []
    labels = []

    for x, y, z in batch:
      input_ids.append(x)
      attention_masks.append(y)
      labels.append(z)

    max_len = max(len(ids) for ids in input_ids)

    padded_input_ids = []
    padded_attention_masks = []
    padded_labels = []

    for ids, attn_mask, lbls in zip(input_ids, attention_masks, labels):
       pad_len = max_len - len(ids)
       
       ids_jax = jnp.array(ids) if not isinstance(ids, jnp.ndarray) else ids
       mask_jax = jnp.array(attn_mask) if not isinstance(attn_mask, jnp.ndarray) else attn_mask
       lbls_jax = jnp.array(lbls) if not isinstance(lbls, jnp.ndarray) else lbls

       padded_input_ids.append(jnp.concatenate([ids_jax, jnp.full((pad_len,), pad_token_id, dtype=ids_jax.dtype)]))
       padded_attention_masks.append(jnp.concatenate([mask_jax, jnp.zeros(pad_len, dtype=mask_jax.dtype)]))
       padded_labels.append(jnp.concatenate([lbls_jax, jnp.full((pad_len,), pad_token_id, dtype=lbls_jax.dtype)]))
    
    stacked_attention_masks = jnp.stack(padded_attention_masks)
    
    if len(stacked_attention_masks.shape) == 2:
        stacked_attention_masks = jnp.expand_dims(stacked_attention_masks, axis=1)
    
    return (
        jnp.stack(padded_input_ids),
        stacked_attention_masks,
        jnp.stack(padded_labels),
    )


In [ ]:
def prm_loss_fn(
    logits, 
    labels, 
    input_ids, 
    placeholder_token_id,
    reward_token_ids
):
    placeholder_mask = input_ids == placeholder_token_id
    logits_at_placeholders = logits[placeholder_mask] 
    labels_at_placeholders = labels[placeholder_mask]  

    logits_reward = logits_at_placeholders[:, reward_token_ids]

    label_indices = jnp.zeros_like(labels_at_placeholders, dtype=jnp.int32)
    for i, token_id in enumerate(reward_token_ids):
        label_indices = jnp.where(labels_at_placeholders == token_id, i, label_indices)

    
    labels_at_placeholders = label_indices.astype(jnp.int32) 
    valid_mask = labels_at_placeholders != -100

    if valid_mask.sum() == 0: return jnp.array(0.0)

    valid_logits = logits_reward[valid_mask]
    valid_labels = labels_at_placeholders[valid_mask]
    
    loss = optax.softmax_cross_entropy_with_integer_labels(valid_logits, valid_labels).mean()
    
    return loss

In [ ]:
#Training Loop
def save_prm_checkpoint(step, params, opt_state, save_dir="prm_checkpoints"):
    os.makedirs(save_dir, exist_ok=True)

    prm_ckpt = {
        "params": params,
        "opt_state": opt_state,
        "step": step
    }

    path = os.path.join(save_dir, f"ckpt_step_{step}.msgpack")

    with open(path, "wb") as f:
        f.write(serialization.to_bytes(prm_ckpt))
    
    print(f"\n✅ Saved checkpoint at step {step} → {path}")


def train_step(batch, graph_def, placeholder_token_id, reward_token_ids, full_state):
    input_ids = jnp.array(batch["input_ids"])
    attention_mask = jnp.array(batch["attention_mask"])
    labels = jnp.array(batch["labels"])

    positions = jnp.broadcast_to(
        jnp.arange(input_ids.shape[1]),
        input_ids.shape
    )

    def loss_fn(state):
        model = nnx.merge(graph_def, state)

        logits, _ = model(
            input_ids,
            positions,
            None,
            attention_mask,
        )

        loss = prm_loss_fn(
            logits=logits,
            input_ids=input_ids,
            labels=labels,
            placeholder_token_id=placeholder_token_id,
            reward_token_ids=reward_token_ids
        )

        return loss
    
    loss, grads = jax.value_and_grad(loss_fn)(full_state)

    filtered_grads = nnx.state(grads, nnx.LoRAParam)

    return loss, filtered_grads


def train_prm(
    dataset, 
    tokenizer, 
    placeholder_token, 
    max_length, 
    reward_tokens,
    batch_size,
    gradient_accumulation_steps,
    num_epochs,
    warmup_ratio,
    learning_rate,
    model,
    pad_token_id
):
    graph_def, full_state = nnx.split(model)
    
    train_data = process_data(
        in_dataset=dataset,
        placeholder_token=placeholder_token,
        in_tokenizer=tokenizer,
        max_length=max_length
    )

    placeholder_token_id = convert_token_to_id(placeholder_token, tokenizer)
    reward_token_ids = [convert_token_to_id(token, tokenizer) for token in reward_tokens]

    num_batches = len(train_data)//batch_size
    num_update_steps_per_epoch = max(1, num_batches // gradient_accumulation_steps)
    total_steps = num_update_steps_per_epoch * num_epochs
    warmup_steps = int(total_steps * warmup_ratio)
    decay_steps = max(1, total_steps - warmup_steps)

    

    optimizer = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.scale_by_adam(b1=0.9, b2=0.95),
        optax.scale_by_schedule(schedule),
        optax.scale(-1.0)
    )

    lora_params = nnx.state(full_state, nnx.LoRAParam)
    opt_state = optimizer.init(lora_params)
    
    params = lora_params

    step = 0
    global_step = 0
    loss_sum = 0.0
    accumulated_grads = None
    current_state = full_state


    for epoch in range(num_epochs):
        print(f"\n=== Epoch {epoch + 1}/{num_epochs} ===")

        np.random.shuffle(train_data)

        epoch_bar = tqdm(
            range(num_batches),
            desc=f"Train Step of epoch {epoch + 1}"
        )

        for batch_idx in epoch_bar:
            batch = padded_data(
                train_data[batch_idx:batch_idx+batch_size],
                pad_token_id=pad_token_id
            )

            input_batch_ids, attention_batch_masks, labels_batch = batch

            loss, lora_grads = train_step(
                {
                    "input_ids": input_batch_ids,
                    "attention_mask": attention_batch_masks,
                    "labels": labels_batch,
                },
                graph_def=graph_def,
                placeholder_token_id=placeholder_token_id,
                reward_token_ids=reward_token_ids,
                full_state=current_state
            )

            print(f"Epoch {epoch+1}, Batch {batch_idx}, loss = {float(loss):.4f}")

            if accumulated_grads is None:
                accumulated_grads = lora_grads
            else:
                accumulated_grads = jax.tree.map(lambda a, b: a + b, accumulated_grads, lora_grads)

            loss_sum += float(loss)
            step += 1

            if step % gradient_accumulation_steps == 0:
                global_step = step // gradient_accumulation_steps
                updates, opt_state = optimizer.update(accumulated_grads, opt_state, params)
                params = optax.apply_updates(params, updates)
                
                temp_model = nnx.merge(graph_def, current_state)
                nnx.update(temp_model, params)
                _, current_state = nnx.split(temp_model)
                del temp_model
                
                avg_loss = loss_sum / gradient_accumulation_steps

                accumulated_grads = jax.tree.map(lambda x: jnp.zeros_like(x), accumulated_grads)
                loss_sum = 0.0

                current_lr = schedule(global_step)
                logs_dict = {
                    "prm_loss": avg_loss,
                    "lr": current_lr,
                }
                epoch_bar.set_postfix(logs_dict)
                print(f"\nStep {global_step}: Loss={avg_loss:.4f}, LR={current_lr:.2e}")

                if global_step % PRM_CHECKPOINT_ENTRY == 0:
                    save_prm_checkpoint(
                        step=global_step,
                        params=params,
                        opt_state=opt_state,
                        save_dir="checkpoints"
                    )
        epoch_bar.close()

    save_prm_checkpoint(
        step=global_step,
        params=params,
        opt_state=opt_state,
        save_dir="checkpoints"
    )
    
    final_model = nnx.merge(graph_def, current_state)
    return final_model

In [ ]:

gc.collect()

trained_model = train_prm(
    dataset=train_data, 
    tokenizer=tokenizer,
    placeholder_token=PRM_PLACEHOLDER_TOKEN,
    max_length=PRM_MAX_LENGTH,
    reward_tokens=PRM_REWARD_TOKENS,
    batch_size=PRM_BATCH_SIZE,
    gradient_accumulation_steps=PRM_GRADIENT_ACCUMULATOR_STEPS,
    num_epochs=PRM_NUM_EPOCHS,
    warmup_ratio=PRM_WARMUP_RATIO,
    learning_rate=PRM_LEARNING_RATE,
    model=lora_policy,
    pad_token_id=PRM_PAD_TOKEN_ID
)



In [ ]:
def evaluate_prm(
    model,
    eval_dataset,
    tokenizer,
    placeholder_token,
    reward_tokens,
    batch_size,
    max_length,
    pad_token_id
):
    model.eval()

    eval_data = process_data(
        in_dataset=eval_dataset,
        placeholder_token=placeholder_token,
        in_tokenizer=tokenizer,
        max_length=max_length
    )

    placeholder_token_id = convert_token_to_id(placeholder_token, tokenizer)
    reward_token_ids = [convert_token_to_id(t, tokenizer) for t in reward_tokens]

    num_batches = len(eval_data) // batch_size
    total_loss = 0.0

    for i in range(num_batches):
        batch = padded_data(
            eval_data[i * batch_size:(i + 1) * batch_size],
            pad_token_id=pad_token_id
        )

        input_ids, attention_mask, labels = batch

        positions = jnp.broadcast_to(
            jnp.arange(input_ids.shape[1]),
            input_ids.shape
        )

        logits, _ = model(
            input_ids,
            positions,
            None,
            attention_mask
        )

        loss = prm_loss_fn(
            logits=logits,
            input_ids=input_ids,
            labels=labels,
            placeholder_token_id=placeholder_token_id,
            reward_token_ids=reward_token_ids
        )

        total_loss += float(loss)

    return total_loss / num_batches


eval_loss = evaluate_prm(
    trained_model,
    eval_data,
    tokenizer,
    PRM_PLACEHOLDER_TOKEN,
    PRM_REWARD_TOKENS,
    PRM_BATCH_SIZE,
    PRM_MAX_LENGTH,
    PRM_PAD_TOKEN_ID
)

print(f"Final Eval Loss: {eval_loss:.4f}")


In [ ]:
match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)

match_format.search(
    f"{reasoning_start}Let me"
    f" think!{reasoning_end}{solution_start}2{solution_end}",
)

Give the model a reward of 3 points if the format matches exactly.

In [ ]:
def match_format_exactly(prompts, completions, **kwargs):
  return [
      0 if match_format.search(response) is None else 3.0
      for response in completions
  ]

We also reward the model if the format of the output matches partially.

In [ ]:
def match_format_approximately(prompts, completions, **kwargs):
  scores = []

  for completion in completions:
    score = 0
    response = completion
    # Count how many keywords are seen - we penalize if too many!
    # If we see 1, then plus some points!
    score += 0.5 if response.count(reasoning_start) == 1 else -0.5
    score += 0.5 if response.count(reasoning_end) == 1 else -0.5
    score += 0.5 if response.count(solution_start) == 1 else -0.5
    score += 0.5 if response.count(solution_end) == 1 else -0.5
    scores.append(score)
  return scores

Reward the model if the answer is correct. A reward is also given if the answer
does not match exactly, i.e., based on how close the answer is to the correct
value.

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
  responses = completions

  extracted_responses = [
      guess.group(1) if (guess := match_format.search(r)) is not None else None
      for r in responses
  ]

  scores = []
  assert len(extracted_responses) == len(
      answer
  ), f"{extracted_responses} and {answer} have mismatching length"
  for guess, true_answer in zip(extracted_responses, answer):
    score = 0
    if guess is None:
      scores.append(0)
      continue
    # Correct answer gets 3 points!
    if guess == true_answer:
      score += 3.0
    # Match if spaces are seen
    elif guess.strip() == true_answer.strip():
      score += 1.5
    else:
      # We also reward it if the answer is close via ratios!
      # Ie if the answer is within some range, reward it!
      try:
        ratio = float(guess) / float(true_answer)
        if ratio >= 0.9 and ratio <= 1.1:
          score += 0.5
        elif ratio >= 0.8 and ratio <= 1.2:
          score += 0.25
        else:
          score -= 1.0  # Penalize wrong answers
      except:
        score -= 0.5  # Penalize
    scores.append(score)
  return scores

Sometimes, the text between `<answer>` and `</answer>` might not be one
number; it can be a sentence. So, we extract the number and compare the answer.

In [ ]:
match_numbers = re.compile(
    rf"{solution_start}.*?([\d\.]{{1,}})", flags=re.MULTILINE | re.DOTALL
)
match_numbers.findall(f"{solution_start}  0.34  {solution_end}")

In [ ]:
def check_numbers(prompts, completions, answer, **kwargs):
  question = kwargs["question"]
  responses = completions

  extracted_responses = [
      guess.group(1) if (guess := match_numbers.search(r)) is not None else None
      for r in responses
  ]

  scores = []
  print("START ============================")
  print(f"Question: {question[0]}")
  print(f"Answer: {answer[0]}")
  print(f"Response: {responses[0]}")
  print(f"Extracted: {extracted_responses[0]}")
  print("END ==============================")
  for guess, true_answer in zip(extracted_responses, answer):
    if guess is None:
      scores.append(0)
      continue
    # Convert to numbers
    try:
      true_answer = float(true_answer.strip())
      guess = float(guess.strip())
      scores.append(1.5 if guess == true_answer else 0.0)
    except:
      scores.append(0)
      continue
  return scores

## Evaluate


Before we train the model, let's evaluate the model on the test set so we can
see the improvement post training.

We evaluate it in two ways:

**Quantitative**

* **Answer Accuracy**: percentage of samples for which the model predicts the
correct final numerical answer  
* **Answer (Partial) Accuracy**: percentage of samples for which the model
predicts a final numerical answer such that the \`model answer / answer\`
ratio lies between 0.9 and 1.1.  
* **Format Accuracy**: percentage of samples for which the model outputs the
correct format, i.e., reasoning between the reasoning special tokens, and the
final answer between the \`\<start\_answer\>\`, \`\<end\_answer\>\` tokens.

**Qualitative**

We'll also print outputs for a few given questions so that we can compare the generated output later.


We define a helper function to generate an answer, given a prompt.

In [ ]:
def generate(
    question, sampler, temperature=0.7, top_k=50, top_p=0.95, seed=None
):
  """Given prompt, generates text."""

  if isinstance(question, str):
    input_batch = [
        TEMPLATE.format(
            system_prompt=SYSTEM_PROMPT,
            question=question,
        ),
    ]
  else:
    input_batch = [
        TEMPLATE.format(
            system_prompt=SYSTEM_PROMPT,
            question=q,
        )
        for q in question
    ]

  out_data = sampler(
      input_strings=input_batch,
      max_generation_steps=768,
      temperature=temperature,
      top_k=top_k,
      top_p=top_p,
      echo=False,
      seed=seed if seed is not None else None,
  )

  output = out_data.text
  if isinstance(question, str):
    return output[0]
  return output

Another helper function for evaluation.

In [ ]:
def evaluate(
    dataset,
    sampler,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_passes=1,
    corr_lst=False,
    make_lst=False,
):
  """Computes accuracy and percentage of outputs matching the format."""

  response_lst = []
  corr = 0
  partially_corr = 0
  corr_format = 0
  total = 0

  for batch in tqdm(dataset):
    answers = batch["answer"]
    questions = batch["question"]
    
    multiple_call_responses = [[] for _ in range(len(questions))]
    for p in range(num_passes):
      responses = generate(
          questions, sampler, temperature, top_k, top_p, seed=p
      )
      for idx, response in enumerate(responses):
        multiple_call_responses[idx].append(response)
    
    for question, multiple_call_response, answer in zip(
        questions, multiple_call_responses, answers
    ):
      # check answer
      corr_ctr_per_question = 0
      partially_corr_per_question = 0
      corr_format_per_question = 0
      for response in multiple_call_response:
        extracted_response = (
            guess.group(1)
            if (guess := match_numbers.search(response)) is not None
            else "-1000000"
        )
        try:
          if float(extracted_response.strip()) == float(answer.strip()):
            corr_ctr_per_question += 1

          ratio = float(extracted_response.strip()) / float(answer.strip())
          if ratio >= 0.9 and ratio <= 1.1:
            partially_corr_per_question += 1
        except:
          print("SKIPPED")

        # check format
        if match_format.search(response) is not None:
          corr_format_per_question += 1

        if (
            corr_ctr_per_question > 0
            and partially_corr_per_question > 0
            and corr_format_per_question > 0
        ):
          break

      if corr_ctr_per_question > 0:
        corr += 1
        if corr_lst and make_lst:
          response_lst.append((question, answer, multiple_call_response))
      else:
        if not corr_lst and make_lst:
          response_lst.append((question, answer, multiple_call_response))
      if partially_corr_per_question > 0:
        partially_corr += 1
      if corr_format_per_question > 0:
        corr_format += 1

      total += 1
      if total % 10 == 0:
        print(
            f"===> {corr=}, {total=}, {corr / total * 100=}, "
            f"{partially_corr / total * 100=}, {corr_format / total * 100=}"
        )

  to_return = (
      corr,
      total,
      corr / total * 100,
      partially_corr / total * 100,
      corr_format / total * 100,
  )
  if make_lst:
    return to_return, response_lst
  return to_return

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=lora_policy,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

Now let's see how the original model does on the test set. You can see the percentages of the mode outputs that are fully correct, partially correct and just correct in format. The following step might take couple of minutes to finish.

In [ ]:
# The evaluation might take up to couple of minutes to finish. Please be patient.

(corr, total, accuracy, partial_accuracy, format_accuracy), resp_lst = evaluate(
    test_dataset,
    sampler,
    **GENERATION_CONFIGS["greedy"],
    corr_lst=False,
    make_lst=True
)
for question, answer, multiple_call_response in resp_lst:
    print(f"question: {question}")
    print(f"answer: {answer}")
    print(f"multiple_call_response")
    for rsp in multiple_call_response:
        print(rsp)
    print("-" * 50)

print(
    f"{corr=}, {total=}, {accuracy=}%, {partial_accuracy=}%,"
    f" {format_accuracy=}%"
)

## Train

Let's set up all the configs first - checkpointing, metric logging and training.
We then train the model.

In [ ]:
# Ckpt saving
checkpointing_options = ocp.CheckpointManagerOptions(
    save_interval_steps=SAVE_INTERVAL_STEPS, max_to_keep=MAX_TO_KEEP
)

# Metrics logger
metrics_logging_options = metrics_logger.MetricsLoggerOptions(
    log_dir="./tensorboard/grpo", flush_every_n_steps=20
)

In [ ]:
# # Logs
# %load_ext tensorboard
# %tensorboard --logdir /tmp/content/tmp/tensorboard/grpo --port=0

In [ ]:
# Optimizer, learning rate scheduler, gradient clipping
optimizer = optax.adamw(
    learning_rate=optax.schedules.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        decay_steps=MAX_STEPS,
        end_value=0.0,
    ),
    b1=B1,
    b2=B2,
    weight_decay=WEIGHT_DECAY,
)
if MAX_GRAD_NORM is not None:
  optimizer = optax.chain(
      optax.clip_by_global_norm(max_norm=MAX_GRAD_NORM),
      optimizer,
  )

In [ ]:
# Training config
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine='vanilla',
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=EVAL_EVERY_N_STEPS,
        max_steps=MAX_STEPS,
        mini_batch_size=TRAIN_MICRO_BATCH_SIZE,
        train_micro_batch_size=TRAIN_MICRO_BATCH_SIZE,
        # metrics logging
        metrics_logging_options=metrics_logging_options,
        # checkpoint saving
        checkpoint_root_directory=CKPT_DIR,
        checkpointing_options=checkpointing_options,
    ),
    rollout_config=base_rollout.RolloutConfig(
        max_tokens_to_generate=TOTAL_GENERATION_STEPS,
        max_prompt_length=MAX_PROMPT_LENGTH,
        kv_cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
    ),
)

grpo_config = GRPOConfig(
    num_generations=NUM_GENERATIONS,
    num_iterations=NUM_ITERATIONS,
    beta=BETA,
    epsilon=EPSILON,
)

### Setting Up the GRPO Trainer

Now we initialize our system for training. First, we create an `RLCluster` instance, which brings together the **policy model (`actor`)**, a **reference model (`reference`)**, and a **tokenizer**. Our `actor` is a trainable LoRA model, while the `reference` is a fixed base model that we use to guide the training.

We then create a `GRPOLearner`, the specialized trainer that uses a list of **reward functions** to evaluate and optimize the model's output, completing the RL training setup.

Tunix trainers are integrated with [Weights & Biases](https://wandb.ai/) to help you visualize the training progress. You can choose how you want to use it:

**Option 1 (Type 1)**: If you're running a quick experiment or just testing things out, choose this. It creates a temporary, private dashboard right in your browser without requiring you to log in or create an account.

**Option 2 (Type 2)**: If you have an existing W&B account and want to save your project's history to your personal dashboard, choose this. You'll be prompted to enter your API key or log in.

In [ ]:
# RL cluster
rl_cluster = rl_cluster_lib.RLCluster(
    actor=lora_policy,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

# GRPO Trainer
grpo_trainer = GRPOLearner(
    rl_cluster=rl_cluster,
    reward_fns=[
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    grpo_config=grpo_config,
)

The first couple of training step might take up to 5 minutes to finish. Please be patient. If you experience long training steps, e.g. >10 minutes per step, please open a bug. Really appreciated!

In [ ]:
with mesh:
  grpo_trainer.train(dataset)

## Evaluate

Let's evaluate our finetuned model!

In [ ]:
import wandb
wandb.init(project='tunix-eval')  # logging bug workaround

In [ ]:
# Load checkpoint first.

trained_ckpt_path = os.path.join(
    CKPT_DIR, "actor", str(MAX_STEPS-1), "model_params"
)

abs_params = jax.tree.map(
    lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype),
    nnx.state(lora_policy, nnx.LoRAParam),
)
checkpointer = ocp.StandardCheckpointer()
trained_lora_params = checkpointer.restore(trained_ckpt_path, target=abs_params)

new_graph_def, new_state = nnx.split(lora_policy)
new_lora_policy = nnx.merge(new_graph_def, new_state)

nnx.update(
    new_lora_policy,
    jax.tree.map(
        lambda a, b: b,
        nnx.state(new_lora_policy, nnx.LoRAParam),
        trained_lora_params,
    ),
)

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=lora_policy,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

In [ ]:
# The evaluation might take up to couple of minutes to finish. Please be patient.

(corr, total, accuracy, partial_accuracy, format_accuracy), resp_lst = evaluate(
    test_dataset,
    sampler,
    **GENERATION_CONFIGS["greedy"],
    corr_lst=False,
    make_lst=True
)
for question, answer, multiple_call_response in resp_lst:
    print(f"question: {question}")
    print(f"answer: {answer}")
    print(f"multiple_call_response")
    for rsp in multiple_call_response:
        print(rsp)
    print("-" * 50)

print(
    f"{corr=}, {total=}, {accuracy=}%, {partial_accuracy=}%,"
    f" {format_accuracy=}%"
)

With sufficient training, you should see that the percentages of correct model outputs have clearly gone up, which means our training worked.